# Clustering K-means sur le dataset Wine Quality

Notebook Jupyter généré automatiquement à partir de votre script Python.

In [ ]:

from ucimlrepo import fetch_ucirepo 
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import PCA


## 1. Chargement du dataset

In [ ]:

wine_quality = fetch_ucirepo(id=186)
X = wine_quality.data.features
y = wine_quality.data.targets


## 2. Nettoyage des valeurs manquantes

In [ ]:

imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)


## 3. Sélection des variables (Mutual Information)

In [ ]:

mi = mutual_info_regression(X, y)
important = pd.Series(mi, index=X.columns).sort_values(ascending=False)
print("Importance des features :\n", important)


## 4. Normalisation

In [ ]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_preprocessed = pd.DataFrame(X_scaled, columns=X.columns)


## 5. Choix du nombre de clusters (Méthodes du coude et silhouette)

In [ ]:

# inertias = []
# K = range(2, 11)

# for k in K:
#     kmeans = KMeans(n_clusters=k, random_state=42)
#     kmeans.fit(X_preprocessed)
#     inertias.append(kmeans.inertia_)

# plt.figure(figsize=(7,5))
# plt.plot(K, inertias, marker='o')
# plt.xlabel("k (nombre de clusters)")
# plt.ylabel("Inertie")
# plt.title("Méthode du coude pour déterminer k")
# plt.grid(True)
# plt.show()

# silhouette_scores = []

# for k in K:
#     kmeans = KMeans(n_clusters=k, random_state=42)
#     labels = kmeans.fit_predict(X_preprocessed)
#     silhouette_scores.append(silhouette_score(X_preprocessed, labels))

# plt.figure(figsize=(7,5))
# plt.plot(K, silhouette_scores, marker='o')
# plt.xlabel("k (nombre de clusters)")
# plt.ylabel("Score de silhouette")
# plt.title("Méthode de la silhouette")
# plt.grid(True)
# plt.show()


## 6. K-means final et visualisation PCA

In [ ]:

best_k = 3
kmeans_final = KMeans(n_clusters=best_k, random_state=42)
labels_final = kmeans_final.fit_predict(X_preprocessed)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_preprocessed)

plt.figure(figsize=(7,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=labels_final, cmap='viridis', s=10)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Clusters K-means projetés en PCA")
plt.colorbar(label="Cluster")
plt.show()


## 7. Analyse des clusters

In [ ]:

X_clustered = X_preprocessed.copy()
X_clustered["cluster"] = labels_final

cluster_profiles = X_clustered.groupby("cluster").mean()

print("\nProfil moyen des clusters (variables normalisées) :")
print(cluster_profiles)


## 8. Comparaison avec la qualité réelle

In [ ]:

comparison = pd.DataFrame({
    "cluster": labels_final,
    "quality": y.values.ravel()
})

print("\nDistribution de la qualité par cluster :")
print(comparison.groupby("cluster")["quality"].describe())

print("\nQualité moyenne par cluster :")
print(comparison.groupby("cluster")["quality"].mean())
